In [1]:
from ipywidgets import Output
output = Output()
output

Output()

## Simulation

In [2]:
from nanover.openmm import OpenMMSimulation
from nanover.mdanalysis.converter import frame_data_to_mdanalysis

simulation = OpenMMSimulation.from_xml_path("../openmm/openmm_files/17-ala.xml")
simulation.load()
universe = frame_data_to_mdanalysis(simulation.make_topology_frame())

## iMD runner + client

In [3]:
from nanover.app import OmniRunner
from nanover.websocket import NanoverImdClient

imd_runner = OmniRunner.with_basic_server(simulation, port=0, name="MOVEABLE RESTRAINTS")
imd_runner.load(0)
imd = imd_runner.app_server.imd

client = NanoverImdClient.from_runner(imd_runner)

In [4]:
with client.root_selection.modify() as selection:
    selection.renderer = "cartoon"

In [5]:
def notify_all(message):
    for command in imd_runner.app_server.commands:
        if "/notify" in command:
            imd_runner.app_server.run_command(command, dict(message=message))

## Restraints

In [6]:
from nanover.app import RenderingSelection
from nanover.imd.imd_state import INTERACTION_PREFIX, ParticleInteraction

RESTRAINT_PREFIX = f"{INTERACTION_PREFIX}MOVEABLE-RESTRAINT."

NEXT_RESTRAINT_INDEX = 0
MOVEABLE_RESTRAINTS: dict[str, ParticleInteraction] = {}
SELECTIONS: dict[str, RenderingSelection] = {}
ACTIVE_RESTRAINTS: set[str] = set()

RENDERER = {
    "render": {
        "particle.scale": .1,
        "bond.scale": 0.0,
        "type": "ball and stick"
    },
    "color": "CornflowerBlue",
}

def add_restraint(indexes):
    global NEXT_RESTRAINT_INDEX
    indexes = [int(index) for index in indexes]

    key = f"{RESTRAINT_PREFIX}{NEXT_RESTRAINT_INDEX}"
    NEXT_RESTRAINT_INDEX += 1
    restraint = ParticleInteraction((0, 0, 0), indexes)
    restraint.scale = 1000
    restraint.interaction_type = "spring"
    MOVEABLE_RESTRAINTS[key] = restraint

    with client.create_selection(key, indexes).modify() as selection:
        selection.renderer = RENDERER
        selection.velocity_reset = True
        selection.interaction_method = "group"
        SELECTIONS[key] = selection

    enable_restraint(key)

    return key


def remove_restraint(key):
    disable_restraint(key)
    client.remove_selection(SELECTIONS[key])
    MOVEABLE_RESTRAINTS.pop(key, None)


def enable_restraint(key):
    if key in ACTIVE_RESTRAINTS:
        return
    ACTIVE_RESTRAINTS.add(key)
    positions = imd_runner.app_server.frame_publisher.current_frame.particle_positions
    restraint = MOVEABLE_RESTRAINTS[key]
    restraint.position = positions[restraint.particles].sum(axis=0) / len(restraint.particles)
    imd.insert_interaction(key, restraint)


def disable_restraint(key):
    if key not in ACTIVE_RESTRAINTS:
        return
    ACTIVE_RESTRAINTS.remove(key)
    imd.remove_interaction(key)

## Commands

In [7]:
def clear_restraints():
    for restraint in list(MOVEABLE_RESTRAINTS.keys()):
        remove_restraint(restraint)
    MOVEABLE_RESTRAINTS.clear()

imd_runner.app_server.register_command("user/restraints/clear", clear_restraints)

In [8]:
from nanover.imd.imd_state import ParticleInteraction

class Mode:
    active = None

    @staticmethod
    def add_to_runner(imd_runner):
        def on_interaction_started(*, key: str, interaction: ParticleInteraction):
            if RESTRAINT_PREFIX in key:
                return
            Mode.active.active.on_interaction_started(key=key, interaction=interaction)

        def on_interaction_stopped(*, key: str, interaction: ParticleInteraction):
            if RESTRAINT_PREFIX in key:
                return
            Mode.active.on_interaction_stopped(key=key, interaction=interaction)

        imd_runner.app_server.imd.interaction_started.add_callback(on_interaction_started)
        imd_runner.app_server.imd.interaction_stopped.add_callback(on_interaction_stopped)
        Mode.enter()

    @classmethod
    def enter(cls):
        Mode.active = cls()
        notify_all(f"ENTERED {cls.__name__}")

    def on_interaction_started(self, *, key: str, interaction: ParticleInteraction):
        pass

    def on_interaction_stopped(self, *, key: str, interaction: ParticleInteraction):
        pass

Mode.add_to_runner(imd_runner)

In [9]:
class InteractMode(Mode):
    def on_interaction_started(self, *, key: str, interaction: ParticleInteraction):
        interacted_indexes = set(interaction.particles)
        for key, restraint in MOVEABLE_RESTRAINTS.items():
            if interacted_indexes.intersection(restraint.particles):
                disable_restraint(key)

    def on_interaction_stopped(self, *, key: str, interaction: ParticleInteraction):
        interacted_indexes = set(interaction.particles)
        for key, restraint in MOVEABLE_RESTRAINTS.items():
            if interacted_indexes.intersection(restraint.particles):
                enable_restraint(key)

imd_runner.app_server.register_command("user/restraints/interact", InteractMode.enter)

In [10]:
class ToggleMode(Mode):
    def on_interaction_started(self, *, key: str, interaction: ParticleInteraction):
        interacted_indexes = set(interaction.particles)
        removed = False
        for key, restraint in MOVEABLE_RESTRAINTS.items():
            if interacted_indexes.intersection(restraint.particles):
                remove_restraint(key)
                removed = True
        if not removed:
            add_restraint(interaction.particles)

imd_runner.app_server.register_command("user/restraints/toggle", ToggleMode.enter)

In [11]:
class InfoMode(Mode):
    def on_interaction_started(self, *, key: str, interaction: ParticleInteraction):
        atom = universe.atoms[interaction.particles[0]]
        notify_all(str(atom))

imd_runner.app_server.register_command("user/info mode", InfoMode.enter)

In [12]:
InteractMode.enter()

In [13]:
from nanover.websocket.record import record_from_runner, BackgroundRecordingContext

RECORDING_PATH = None
RECORDING_COUNT = 0
RECORDER: BackgroundRecordingContext | None = None

def start_recording():
    global RECORDER, RECORDING_COUNT, RECORDING_PATH
    RECORDING_PATH = f"RECORDING-{RECORDING_COUNT}-{imd_runner.simulation.name}.nanover.zip"
    RECORDER = record_from_runner(imd_runner, RECORDING_PATH)
    RECORDING_COUNT += 1
    notify_all(f"STARTED RECORDING to {RECORDING_PATH}")

def stop_recording():
    if RECORDER is not None:
        RECORDER.close()
        notify_all(f"FINISHED RECORDING to {RECORDING_PATH}")

imd_runner.app_server.register_command("user/recording/start", start_recording)
imd_runner.app_server.register_command("user/recording/stop", stop_recording)

In [ ]:
from nanover.utilities.state_dictionary import DictionaryChange

NEXT_CHECKPOINT_INDEX = 0

def mark_checkpoint():
    global NEXT_CHECKPOINT_INDEX
    imd_runner.app_server.update_state(DictionaryChange(updates={"mark.checkpoint": NEXT_CHECKPOINT_INDEX}))
    NEXT_CHECKPOINT_INDEX += 1

imd_runner.app_server.register_command("user/mark/checkpoint", mark_checkpoint)